In [1]:
import tempfile
from sklearn.datasets import fetch_openml

# Use a fresh data_home to avoid md5 mismatches from stale cached downloads
survey = fetch_openml(data_id=534, as_frame=True, data_home=tempfile.mkdtemp())

X = survey.data[survey.feature_names]

In [2]:
import skrub

skrub.TableReport(X)

Processing column  10 / 10


,,,,,,,,,,


In [3]:
import pandas as pd

X = pd.get_dummies(X, drop_first=True)
X

,EDUCATION,EXPERIENCE,AGE,SOUTH_yes,SEX_male,UNION_not_member,RACE_Other,RACE_White,OCCUPATION_Management,OCCUPATION_Other,OCCUPATION_Professional,OCCUPATION_Sales,OCCUPATION_Service,SECTOR_Manufacturing,SECTOR_Other,MARR_Unmarried
0,8,21,35,False,False,True,False,False,False,True,False,False,False,True,False,False
1,9,42,57,False,False,True,False,True,False,True,False,False,False,True,False,False
2,12,1,19,False,True,True,False,True,False,True,False,False,False,True,False,True
3,12,4,22,False,True,True,False,True,False,True,False,False,False,False,True,True
4,12,17,35,False,True,True,False,True,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529,18,5,29,False,True,True,False,True,False,False,True,False,False,False,True,True
530,12,33,51,False,False,True,True,False,False,False,True,False,False,False,True,False
531,17,25,48,False,False,False,True,False,False,False,True,False,False,False,True,False
532,12,13,31,True,True,False,False,True,False,False,True,False,False,False,True,False


In [4]:
y = survey.target.values.ravel()
y

array([ 5.1 ,  4.95,  6.67,  4.  ,  7.5 , 13.07,  4.45, 19.47, 13.28,
        8.75, 11.35, 11.5 ,  6.5 ,  6.25, 19.98,  7.3 ,  8.  , 22.2 ,
        3.65, 20.55,  5.71,  7.  ,  3.75,  4.5 ,  9.56,  5.75,  9.36,
        6.5 ,  3.35,  4.75,  8.9 ,  4.  ,  4.7 ,  5.  ,  9.25, 10.67,
        7.61, 10.  ,  7.5 , 12.2 ,  3.35, 11.  , 12.  ,  4.85,  4.3 ,
        6.  , 15.  ,  4.85,  9.  ,  6.36,  9.15, 11.  ,  4.5 ,  4.8 ,
        4.  ,  5.5 ,  8.4 ,  6.75, 10.  ,  5.  ,  6.5 , 10.75,  7.  ,
       11.43,  4.  ,  9.  , 13.  , 12.22,  6.28,  6.75,  3.35, 16.  ,
        5.25,  3.5 ,  4.22,  3.  ,  4.  , 10.  ,  5.  , 16.  , 13.98,
       13.26,  6.1 ,  3.75,  9.  ,  9.45,  5.5 ,  8.93,  6.25,  9.75,
        6.73,  7.78,  2.85,  3.35, 19.98,  8.5 ,  9.75, 15.  ,  8.  ,
       11.25, 14.  , 10.  ,  6.5 ,  9.83, 18.5 , 12.5 , 26.  , 14.  ,
       10.5 , 11.  , 12.47, 12.5 , 15.  ,  6.  ,  9.5 ,  5.  ,  3.75,
       12.57,  6.88,  5.5 ,  7.  ,  4.5 ,  6.5 , 12.  ,  5.  ,  6.5 ,
        6.8 ,  8.75,

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [6]:
from tabicl import TabICLRegressor

clf = TabICLRegressor(n_estimators=2, device="cpu")  # use n_estimators=8 for best results
clf.fit(X_train, y_train)

,n_estimators,2
,norm_methods,None
,feat_shuffle_method,'latin'
,outlier_threshold,4.0
,batch_size,8
,kv_cache,False
,model_path,None
,allow_auto_download,True
,checkpoint_version,'tabicl-regressor-v2-20260212.ckpt'
,device,'cpu'
,use_amp,'auto'


In [7]:
preds = clf.predict(X_test)

In [8]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold

models = {
    "Random Forest": RandomForestRegressor(
        min_samples_leaf=5, random_state=0, n_jobs=-1
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_leaf_nodes=15, random_state=0, early_stopping=False
    ),
}
param_grids = {
    "Random Forest": {
        "n_estimators": [10, 20, 50, 100, 200],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None],
    },
    "Hist Gradient Boosting": {
        "max_iter": [10, 20, 50, 100, 300, 500],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "max_depth": [None, 3, 5, 10],
        "max_leaf_nodes": [15, 31, 63],
        "min_samples_leaf": [10, 20, 50],
        "l2_regularization": [0.0, 0.1, 1.0],
    },
}
cv = KFold(n_splits=4, shuffle=True, random_state=0)

results = {}
for name, model in models.items():
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        return_train_score=True,
        cv=cv,
    ).fit(X_train, y_train)
    model.set_params(**grid_search.best_params_)
    model.fit(X_train, y_train)
    results[name] = model.predict(X_test)

In [9]:
from sklearn.dummy import DummyRegressor

dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train, y_train)
dummy_pred = dummy_regr.predict(X_test)

In [10]:
from sklearn.metrics import mean_squared_error

print('Tabular')
print(mean_squared_error(y_test, preds))
print('Dummy')
print(mean_squared_error(y_test, dummy_pred))
print('Tuned RF')
print(mean_squared_error(y_test, results['Random Forest']))
print('Tuned Hist Gradient Boosting')
print(mean_squared_error(y_test, results['Hist Gradient Boosting']))

Tabular
13.643657569370388
Dummy
18.43914591487873
Tuned RF
13.087218148745993
Tuned Hist Gradient Boosting
13.85243084774102
